In [ ]:
#%%
# Installing modules
%pip install openpyxl

from parcels import (
    AdvectionRK4_3D,
    FieldSet,
    JITParticle,
    ParticleSet,
    Variable
)
from operator import attrgetter
from pathlib import Path
import datetime
import numpy as np
import xarray as xr
from matplotlib import colormaps as cm
from matplotlib import pyplot as plt
import cartopy
import sys
import pandas as pd
import hvplot.pandas
import geoviews
import random

#%% Define Fieldset
data_path = "/home/jovyan/shared_materials/model_data/VIKING20X.L46-KKG36107B/"
mesh_file = "/home/jovyan/shared_materials/model_data/mask/VIKING20X.L46-KKG36107B/1_mesh_mask.nc"

U_files = sorted(Path(data_path).glob("1_VIKING*_1m_199*_grid_U.nc"))
V_files = sorted(Path(data_path).glob("1_VIKING*_1m_199*_grid_V.nc"))
W_files = sorted(Path(data_path).glob("1_VIKING*_1m_199*_grid_W.nc_repaire_depthw_time"))
T_files = sorted(Path(data_path).glob("1_VIKING*_1m_199*_grid_T.nc"))

def fieldset_definitions(
    list_of_filenames_U, list_of_filenames_V, list_of_filenames_W,list_of_filenames_T, mesh_mask
):

    filenames = {
        "U": {
            "lon": (mesh_mask),
            "lat": (mesh_mask),
            "depth": list_of_filenames_W[0],
            "data": list_of_filenames_U,
        },
        "V": {
            "lon": (mesh_mask),
            "lat": (mesh_mask),
            "depth": list_of_filenames_W[0],
            "data": list_of_filenames_V,
        },
        "W": {
            "lon": (mesh_mask),
            "lat": (mesh_mask),
            "depth": list_of_filenames_W[0],
            "data": list_of_filenames_W,
        },
        "T": {
            "lon": (mesh_mask),
            "lat": (mesh_mask),
            "depth": list_of_filenames_T[0],
            "data": list_of_filenames_T,
        },
    }

    variables = {
        "U": "vozocrtx",
        "V": "vomecrty",
        "W": "vovecrtz",
        "T": "votemper",
    }

    dimensions = {
        "U": {
            "lon": "glamf",
            "lat": "gphif",
            "depth": "depthw",
            "time": "time_counter",
        },
        "V": {
            "lon": "glamf",
            "lat": "gphif",
            "depth": "depthw",
            "time": "time_counter",
        },
        "W": {
            "lon": "glamf",
            "lat": "gphif",
            "depth": "depthw",
            "time": "time_counter",
        },
        "T": {
            "lon": "glamf",
            "lat": "gphif",
            "depth": "deptht",
            "time": "time_counter",
        },
    }

    return FieldSet.from_nemo(
        filenames,
        variables,
        dimensions,
        # indices=indices,
        mesh="spherical",
        tracer_interp_method="cgrid_tracer",
        allow_time_extrapolation=False,
    )

fieldset = fieldset_definitions(
    list_of_filenames_U=U_files,
    list_of_filenames_V=V_files,
    list_of_filenames_W=W_files,
    list_of_filenames_T=T_files,
    mesh_mask=mesh_file,
)           
   
#%% This is the file with the core information
file_path = Path("/home/jovyan//my_materials/2024-10_parcels-workshop/notebooks/2024_foram_drift/data/Tierney_2019_SOM1_Atlantic.xlsx")
df = pd.read_excel(file_path)
#df

"""
df.hvplot.points(
    x="longitude",
    y="latitude",
    c="size_lower",
    geo=True,
    coastline=True,
)
                        # Map of cores
df.hvplot.points(
    x="longitude",
    y="latitude",
    c="species",
    geo=True,
    coastline=True,
)
"""

#%% Foram properties
foramkey =	{  "pachy": 1,  "incompta": 2,  "bulloides": 3,  "sacculifer": 4,  "ruber_w": 5,  "ruber_p": 6  }

foramvsink = {"pachy" : {"min" : 0.0035,"max" : 0.016},  "incompta" : {"min" : 0.0035,"max" : 0.016},  "bulloides" : {"min" : 0.0035,"max" : 0.016},
              "sacculifer" : {"min" : 0.0035,"max" : 0.016},  "ruber_w" : {"min" : 0.0035,"max" : 0.016},  "ruber_p" : {"min" : 0.0035,"max" : 0.016}} 
                # in m/s
foramhabdepth = {"pachy" : {"min" : 50,"max" : 50},  "incompta" : {"min" : 50,"max" : 50},  "bulloides" : {"min" : 50,"max" : 50},
              "sacculifer" : {"min" : 50,"max" : 50},  "ruber_w" : {"min" : 50,"max" : 50},  "ruber_p" : {"min" : 50,"max" : 50}} 
                # in m
foramlifetime =  {  "pachy": 30,  "incompta": 30,  "bulloides": 30,  "sacculifer": 30,  "ruber_w": 15,  "ruber_p": 15  }
                # in days
                
#%% Begin loop here
# for i in range (0, np.max(df)+1)
    #i = 304
    rng = np.random.default_rng(seed=i)
    # Particle locations randomized within radius degree around core coordinates
    number_particles = 5000
    radius = 0.005 # degree ~500m
    lon_bds = df["longitude"][i] 
    lat_bds = df["latitude"][i] 
    r = radius * np.sqrt(rng.uniform(size=(number_particles)))
    theta = rng.uniform(size=(number_particles)) * 2 * np.pi
    # (x – xM)² + (y – yM)² = r²
    lon = lon_bds + r * np.cos(theta)
    lat = lat_bds + r * np.sin(theta)
    depth = df["wdepth"][i] * np.ones(number_particles) # Coretop water depth
    # m
    vSink = foramvsink[df["species"][i]]["max"] * np.ones(number_particles)# = 1400 m/d ... 300 m/d ... # Way to switch between min and max?
    # m/s
    #species = foramkey[df["species"][i]] * np.ones(number_particles) # allready recorded in metadata + filename
    hab_depth = foramhabdepth[df["species"][i]]["max"] * np.ones(number_particles) # Way to switch between min and max?
    # m
    time = np.mean(fieldset.U.grid.time) #time = fieldset.U.grid.time[-1] # end of fieldset data
    # Simulation
    runtime_in_days = 182 # foramlifetime[df["species"]][i] + ceil(df["wdepth"][i] / (foramvsink[df["species"][i]]["max"] * 86400)) # for one release?
    dt_in_hours = 0.1 #360s/ 6min Set to run backwards in time, change at execution down below
    hash = 4 #random.getrandbits(128) what is this for?
    outputfilename = df["corename"][i] + "_" + df["species"][i] + "_" +str(i) + "_" + str("%032x" % hash) + "_" + ".zarr"
    outputdt_hours = 12
    
    #%% Define ParticleSet
    extra_vars = [
        Variable("temperature", dtype=np.float32, initial= -999
        ),
        Variable("sinkingSpeed", dtype=np.float32
        ),
        Variable("habDepth", dtype=np.float32
        ),
    ]
    
    # Records Temperature
    TSampler = JITParticle.add_variables(extra_vars)
    
    pset = ParticleSet.from_list(
        fieldset = fieldset,
        pclass = TSampler,
        lon = lon,
        lat = lat,
        depth = depth,
        time = time * np.ones(np.size(lon)),
        repeatdt = datetime.timedelta(days=1), # Maybe change
        sinkingSpeed = vSink,
        habDepth = hab_depth
    )
    
    #%% Define Kernels
    def CheckError(particle, fieldset, time):
        if particle.state >= 50:  # This captures all Errors
            particle.delete()
            
    def SampleT(particle, fieldset, time): # Interpolates particle temperature from Fieldset temperature
        particle.temperature = fieldset.T[time, particle.depth, particle.lat, particle.lon]
    
    def ActiveSinking(particle,fieldset,time): 
        if particle.depth <= particle.habDepth:
            particle.sinkingSpeed = 0 # Particles do not sink/rise when they reach habitat depth, just drift 
           # particle.dt =  3600 # s
        particle_ddepth += particle.sinkingSpeed * particle.dt
        
    #%% Outputfile
    outputfile = pset.ParticleFile(
        name=outputfilename, 
        outputdt=datetime.timedelta(hours=outputdt_hours),
        chunks=(10000,48) # More? Less?
    )
    
    outputfile.metadata["species"] = str(df["species"][i])
    outputfile.metadata["vsink"] = str(foramvsink[df["species"][i]]["max"])
    outputfile.metadata["habdepth"] = str(hab_depth[0])
    
    #%% Execute Kernels
    pset.execute(
        [SampleT,AdvectionRK4_3D,ActiveSinking,CheckError],
        runtime= datetime.timedelta(days=runtime_in_days),
        dt= -datetime.timedelta(hours=dt_in_hours), # note the minus
        output_file =  outputfile
    )
    
    #ds = xr.open_zarr(outputfilename).compute()
    #ds # Opens file